# Aula 10 — Sistema de Partículas com Blending Aditivo

Exemplo de VFX em OpenGL clássico, com foco em arquitetura limpa e manutenção.

## Objetivos
- Implementar emissor de partículas com parâmetros configuráveis.
- Atualizar física e ciclo de vida das partículas.
- Renderizar com blending aditivo para efeito de brilho.

## Conceitos-chave
- `Particle` + `ParticleEmitter` com responsabilidades separadas.
- `alpha` baseado na vida restante.
- `glBlendFunc(GL_SRC_ALPHA, GL_ONE)` para composição aditiva.

## Controles
- `Espaço`: pausa/retoma.
- `C`: limpa partículas.
- `+` / `-`: ajusta taxa de emissão.
- `ESC`: sair.

In [1]:
from dataclasses import dataclass
from random import Random
from math import cos, sin, tau
from OpenGL.GL import *
from OpenGL.GLUT import *

@dataclass
class Particle:
    """Partícula elementar do efeito."""
    x: float
    y: float
    vx: float
    vy: float
    life: float
    max_life: float
    size: float

class ParticleEmitter:
    """Gerencia emissão, update e render de partículas."""
    def __init__(self) -> None:
        self.rng = Random(7)
        self.particles: list[Particle] = []
        self.emit_rate = 35
        self.gravity = -0.0007
        self.paused = False

    def clear(self) -> None:
        self.particles.clear()

    def _spawn_one(self) -> None:
        angle = self.rng.uniform(-0.8, 0.8)
        speed = self.rng.uniform(0.008, 0.018)
        vx = speed * angle
        vy = self.rng.uniform(0.010, 0.022)
        life = self.rng.uniform(0.8, 1.4)
        size = self.rng.uniform(0.010, 0.025)
        self.particles.append(Particle(0.0, -0.85, vx, vy, life, life, size))

    def update(self) -> None:
        if self.paused:
            return

        for _ in range(self.emit_rate):
            self._spawn_one()

        alive = []
        for p in self.particles:
            p.vy += self.gravity
            p.x += p.vx
            p.y += p.vy
            p.life -= 0.016
            if p.life > 0.0:
                alive.append(p)
        self.particles = alive

    def _disc(self, x: float, y: float, r: float, a: float) -> None:
        glColor4f(1.0, 0.55, 0.12, a)
        glBegin(GL_POLYGON)
        for i in range(20):
            t = tau * i / 20
            glVertex2f(x + r * cos(t), y + r * sin(t))
        glEnd()

    def render(self) -> None:
        glClear(GL_COLOR_BUFFER_BIT)
        glLoadIdentity()
        for p in self.particles:
            alpha = max(0.0, min(1.0, p.life / p.max_life))
            self._disc(p.x, p.y, p.size, alpha)
        glutSwapBuffers()

emitter = ParticleEmitter()

def configure_close_behavior() -> bool:
    try:
        glutSetOption(GLUT_ACTION_ON_WINDOW_CLOSE, GLUT_ACTION_GLUTMAINLOOP_RETURNS)
        return True
    except Exception:
        return False

def display() -> None: emitter.render()

def keyboard(key: bytes, _x: int, _y: int) -> None:
    if key == b' ': emitter.paused = not emitter.paused
    elif key in (b'c', b'C'): emitter.clear()
    elif key == b'+': emitter.emit_rate = min(emitter.emit_rate + 5, 120)
    elif key == b'-': emitter.emit_rate = max(emitter.emit_rate - 5, 5)
    elif key == b'\x1b': raise SystemExit('ESC')

def tick(_v: int) -> None:
    emitter.update()
    glutPostRedisplay()
    glutTimerFunc(16, tick, 0)

In [ ]:
glutInit()
close_safe = configure_close_behavior()
glutInitDisplayMode(GLUT_DOUBLE | GLUT_RGB)
glutInitWindowSize(1200, 850)
glutCreateWindow(b"Aula 10 - Sistema de Particulas")
glClearColor(0.02, 0.02, 0.05, 1.0)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE)
glutDisplayFunc(display)
glutKeyboardFunc(keyboard)
glutTimerFunc(0, tick, 0)

try:
    glutMainLoop()
except SystemExit:
    print("GLUT chamou SystemExit ao fechar a janela.")

print("Kernel preservado:", close_safe)